# Scalable ML Lab 1 — Colab T4 环境初始化与 Smoke Test

本 Notebook 用于在 **Google Colab (T4 GPU)** 上一键完成：
1. 挂载团队共享 Google Drive（[共享云盘入口](https://drive.google.com/drive/folders/1eRebAfEqnmBqUf3MgwX3NlcAFC54Izzp?usp=sharing)）
2. 拉取公开 GitHub 仓库 `https://github.com/Mrpooool/ScalableMlLab.git` 并挂载 `lab1/src/`
3. 下载 CIFAR-10（高速镜像）并生成/核对固定的 `45,000 / 5,000` 划分 (`SPLIT_SEED=2223`)
4. 检查手写 Part A9 **AlexNet** 与 **ResNet-18** 的各阶段特征图尺寸与参数量
5. 运行 **1-epoch Smoke Test**（验证训练、确定性 `train_eval`、峰值显存、推理测速、checkpoint 写入 Drive 与标准 JSON 落盘）

> **首次使用提示（三位同学通用）**：
> - 菜单选择 **代码执行程序 → 更改运行时类型 → T4 GPU**
> - 队友首次打开 [共享云盘链接](https://drive.google.com/drive/folders/1eRebAfEqnmBqUf3MgwX3NlcAFC54Izzp?usp=sharing) 后，点击顶部文件夹名旁下拉箭头 → **整理 → 添加快捷方式到云端硬盘 → 我的云端硬盘（My Drive）**，确保下方 `DRIVE_FOLDER_NAME` 与你云端硬盘里的文件夹名称一致。

In [ ]:
# 1. 挂载 Google Drive 并检查共享目录
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

# 如果你的共享云盘文件夹名称不是 ScalableMlLab，只需修改这一行：
DRIVE_FOLDER_NAME = "ScalableMlLab"
SHARED_DRIVE_URL = "https://drive.google.com/drive/folders/1eRebAfEqnmBqUf3MgwX3NlcAFC54Izzp?usp=sharing"

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
if not DRIVE_ROOT.exists():
    print(f"[提示] 未在 MyDrive 下找到 '{DRIVE_FOLDER_NAME}'，当前 MyDrive 顶层目录如下：")
    for p in sorted(Path("/content/drive/MyDrive").iterdir()):
        if p.is_dir():
            print("  -", p.name)
    raise FileNotFoundError(
        f"请先打开 {SHARED_DRIVE_URL} 并添加快捷方式到 My Drive，或将 DRIVE_FOLDER_NAME 改为上方实际文件夹名。"
    )

DRIVE_LAB1 = DRIVE_ROOT / "lab1"
for sub in ("data", "checkpoints", "results", "configs"):
    (DRIVE_LAB1 / sub).mkdir(parents=True, exist_ok=True)

print("Shared Drive lab1 directory ready:", DRIVE_LAB1)

In [ ]:
# 2. 拉取公开 GitHub 仓库并将 lab1/src 加入 Python 路径
REPO_URL = "https://github.com/Mrpooool/ScalableMlLab.git"
REPO_DIR = Path("/content/ScalableMlLab")
BRANCH = "main"  # 个人分支开发时可改为 "feat/lab1-role-a" 等

if not REPO_DIR.exists():
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull

SRC_DIR = REPO_DIR / "lab1" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(REPO_DIR)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[警告] 当前未分配 GPU，请在菜单中切换运行时类型为 T4 GPU！")

In [ ]:
# 3. 加载 CIFAR-10 并生成/核对固定 train/val 划分 (45,000 / 5,000, SPLIT_SEED=2223)
import shutil
import pandas as pd

from data import CIFAR10_CLASSES, get_or_create_split, load_cifar10_datasets, make_loaders
from io_utils import load_json

train_ds, test_ds = load_cifar10_datasets(DRIVE_LAB1 / "data")

SPLIT_PATH = REPO_DIR / "lab1" / "configs" / "split_indices.json"
split_info = get_or_create_split(train_ds, SPLIT_PATH)

# 同步备份一份到 Drive，方便查看或回传仓库
shutil.copy2(SPLIT_PATH, DRIVE_LAB1 / "configs" / "split_indices.json")

stats = split_info["stats"]
train_ids = split_info["train_ids"]
val_ids = split_info["val_ids"]

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ds)}")
display(pd.DataFrame(stats, index=["red", "green", "blue"]))

In [ ]:
# 4. 验证手写 Part A9 AlexNet 与 ResNet-18 的特征图尺寸与参数量
from models import build_model, count_parameters, trace_shapes

for model_name in ("alexnet_cifar", "resnet18_cifar"):
    model, input_size = build_model(model_name, device=device)
    params = count_parameters(model)
    shapes = trace_shapes(model, input_size=input_size, device=device)
    print(f"\n=== {model_name} (input={input_size}x{input_size}) ===")
    print("Parameters:", params)
    display(pd.DataFrame(shapes))
    del model

In [ ]:
# 5. 运行 1-epoch Smoke Test（仅验证全链路，不评测 test 集，不污染正式结果）
from train import ExperimentConfig, run_experiment

SMOKE_RESULTS_DIR = DRIVE_LAB1 / "results" / "smoke_test"
SMOKE_CKPT_DIR = DRIVE_LAB1 / "checkpoints" / "smoke_test"
SMOKE_MANIFEST = SMOKE_RESULTS_DIR / "checkpoints_manifest.json"

baseline_cfg = load_json(REPO_DIR / "lab1" / "configs" / "baseline.json")
loaders = make_loaders(
    train_dataset=train_ds,
    test_dataset=test_ds,
    train_ids=train_ids,
    val_ids=val_ids,
    batch_size=baseline_cfg["batch_size"],
    seed=baseline_cfg["default_seed"],
    num_workers=2,
)

smoke_summaries = []
for model_name in ("alexnet_cifar", "resnet18_cifar"):
    mcfg = baseline_cfg["baselines"][model_name]
    cfg = ExperimentConfig(
        exp_id=f"smoke_{model_name}_e1_s42",
        task="phase1_smoke",
        owner="role_a",
        model=model_name,
        model_options=mcfg["model_options"],
        seed=baseline_cfg["default_seed"],
        epochs=1,
        batch_size=baseline_cfg["batch_size"],
        optimizer=mcfg["optimizer"],
        lr=mcfg["lr"],
        momentum=mcfg["momentum"],
        weight_decay=mcfg["weight_decay"],
        scheduler=mcfg["scheduler"],
        notes="Phase 1 1-epoch smoke test on Colab T4",
    )
    _, record, _ = run_experiment(
        config=cfg,
        loaders=loaders,
        stats=stats,
        device=device,
        results_dir=SMOKE_RESULTS_DIR,
        checkpoint_dir=SMOKE_CKPT_DIR,
        manifest_path=SMOKE_MANIFEST,
        drive_rel_prefix="checkpoints/smoke_test",
        evaluate_test=False,
        run_benchmark=True,
        repo_root=REPO_DIR,
    )
    smoke_summaries.append(
        {
            "exp_id": record["exp_id"],
            "device": record["environment"]["device_name"],
            "params": record["model_stats"]["total_params"],
            "epoch_sec": record["training"]["avg_epoch_time_sec"],
            "peak_mem_mb": record["training"]["peak_gpu_mem_mb"],
            "val_acc_e1": round(record["training"]["best_val_acc"], 4),
            "bs1_ms": round(record["inference_benchmark"]["bs1_latency_ms"], 3),
            "bs128_imgs_s": round(record["inference_benchmark"]["bs128_throughput_imgs_per_sec"], 1),
        }
    )

display(pd.DataFrame(smoke_summaries))
print("Smoke test JSONs saved under:", SMOKE_RESULTS_DIR)